In [1]:
import sys
sys.path.insert(0,'..')
from warnings import filterwarnings
filterwarnings("ignore")
%load_ext autoreload
%autosave 180

Autosaving every 180 seconds


In [2]:
%autoreload
import os
import random
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from apex import amp
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader
from source.version6.data import trainLoader
from source.version6.model import EfficientModel
from source.version6.train import trainModel
from source.version6.loss import BCELoss
from catalyst.data.sampler import BalanceClassSampler

In [3]:
SEED = 42

def seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
    return None

seed(42)

In [4]:
def train(fold):
    loader = {}
    loader['image_path'] = '../../data/cdeotte/train/train/'
    loader['label_path'] = '../../data/cdeotte/data.csv'
    loader['fold_idx'] = fold
    train, valid = trainLoader(**loader)
    params = {}
    params['batch_size'] = 10
    params['num_workers'] = 4
    params['drop_last'] = True
    train = DataLoader(train, shuffle=True, **params)
    valid = DataLoader(valid, **params)
    model = EfficientModel()
    weights = '../../model/pretrain/effnet-b4/model_{}.pt'.format(fold)
    weights = torch.load(weights, map_location='cpu')
    model.load_state_dict(weights['model_state_dict'])
    model = model.to('cuda:0')
    optimizer = AdamW(model.parameters(), lr=1e-05, weight_decay=0.)
    schedular = ReduceLROnPlateau(optimizer, factor=0.5, patience=0, min_lr=1e-8)
    model, optimizer = amp.initialize(model, optimizer, opt_level='O2', verbosity=False)
    trainer = {}
    trainer['model'] = model
    trainer['train_data'] = train
    trainer['valid_data'] = valid
    trainer['loss_fn'] = BCELoss()
    trainer['optimizer'] = optimizer
    trainer['save_path'] = '../../model/version6/model_{}.pt'.format(fold)
    trainer['epochs'] = 15
    trainer['batch'] = 10
    trainer['scheduler'] = schedular
    trainModel(**trainer)
    model.cpu()
    del model
    return None

In [ ]:
train(0)

Train Images: 35419 Valid Images: 6527
Loaded pretrained weights for efficientnet-b4


100% 35410/35410 [23:25<00:00, 25.20it/s, trn_ls=0.6507, val_ls=0.2544, val_mt=0.8813]
100% 35410/35410 [23:22<00:00, 25.25it/s, trn_ls=0.4089, val_ls=0.2102, val_mt=0.9020]
100% 35410/35410 [23:12<00:00, 25.42it/s, trn_ls=0.3694, val_ls=0.1816, val_mt=0.9124]
 81% 28840/35410 [18:34<04:17, 25.50it/s, trn_ls=0.34390]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

100% 35410/35410 [23:28<00:00, 25.15it/s, trn_ls=0.3164, val_ls=0.1640, val_mt=0.9234]
 26% 9340/35410 [05:55<17:10, 25.29it/s, trn_ls=0.29860]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

In [ ]:
train(1)

Train Images: 35397 Valid Images: 6535
Loaded pretrained weights for efficientnet-b4


 42% 14990/35390 [09:32<12:40, 26.84it/s, trn_ls=0.86700]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

100% 35390/35390 [23:30<00:00, 25.09it/s, trn_ls=0.6427, val_ls=0.2698, val_mt=0.8819]
 21% 7280/35390 [04:36<18:31, 25.30it/s, trn_ls=0.44190]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

 76% 26820/35390 [16:55<05:34, 25.63it/s, trn_ls=0.42130]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the 

In [ ]:
train(2)

Train Images: 35389 Valid Images: 6565
Loaded pretrained weights for efficientnet-b4


100% 35380/35380 [23:25<00:00, 25.18it/s, trn_ls=0.6442, val_ls=0.2575, val_mt=0.8747]
100% 35380/35380 [23:27<00:00, 25.14it/s, trn_ls=0.4112, val_ls=0.2249, val_mt=0.8931]
 60% 21330/35380 [13:39<08:49, 26.53it/s, trn_ls=0.37480]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

100% 35380/35380 [23:46<00:00, 24.81it/s, trn_ls=0.3671, val_ls=0.2046, val_mt=0.9005]
 19% 6590/35380 [04:14<17:29, 27.43it/s, trn_ls=0.33820]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_l

In [ ]:
train(3)

 76% 26750/35390 [16:59<05:16, 27.32it/s, trn_ls=0.71830]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

100% 35390/35390 [23:29<00:00, 25.11it/s, trn_ls=0.4139, val_ls=0.2119, val_mt=0.8947]
 31% 11130/35390 [07:00<14:52, 27.19it/s, trn_ls=0.38340]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

 87% 30960/35390 [19:30<02:56, 25.08it/s, trn_ls=0.37080]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the

In [ ]:
train(4)

Train Images: 35419 Valid Images: 6519
Loaded pretrained weights for efficientnet-b4


 42% 14990/35410 [09:33<12:51, 26.46it/s, trn_ls=0.87470]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

 99% 35000/35410 [22:18<00:15, 25.67it/s, trn_ls=0.64940]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

 55% 19610/35410 [12:37<10:00, 26.30it/s, trn_ls=0.41930]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`-